# Imports

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
from random import sample
from cellface.storage.container import Raw
from ovizioapi.compute import capture_to_raw
from ovizioapi.metadata import get_creation_date, get_number_of_images
from tqdm.dask import TqdmCallback

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def get_initials(fullname: str) -> str:
    names = fullname.split(" ")
    initials = "".join(n[0].upper() for n in names)
    return initials 

# Source

In [ ]:
# Path where your Captures are stored currently
base_path = Path(r"D:\cellface")
# Number of the Capture you want to process
cpt_nr = 16
# Capture you want to containerize
source = base_path / f"Capture {cpt_nr}.h5"
n_images = get_number_of_images(source)

display(f"Source File: {source}")
display(f"Found images in source file: {n_images}")

# Meta Data

In [ ]:
creation_date = get_creation_date(source)
date_string = creation_date.strftime("%Y-%m-%d_%H-%M-%S")

md = {
    'capture': cpt_nr,
    'donor': 'Unknown',
    'experimenter': 'Christian Klenk',
    'experimenterID': 'ga76quz',
    'notes': 'Demo Measurement',
    'sampleType': 'WB',
    'project': 'Demo',
}

# Destination

In [ ]:
file_name = ""
file_name += f"{md['project']}_"
file_name += f"{md['sampleType']}_"
file_name += f"{md['donor']}_"
file_name += f"{date_string}_"
file_name += f"{get_initials(md['experimenter'])}_"
file_name += f"Capture_{md['capture']}"
file_name += ".raw"

destination = base_path / file_name
#destination = Path(r"Z:\cellface\Data\other") / file_name
display(f"Source File:      {source}")
display(f"Destintaion File: {destination}")
display("Meta Data Information")
display(md)

When you are happy with the settings above ...

# Let's Go!

In [ ]:

# Start Containering
capture_to_raw(source, destination, md, callback=TqdmCallback)

# Inspect the created Container

## Check tree structure

In [ ]:
with Raw(destination, mode='r') as raw:
    raw.tree(level=4)

## Check Meta Data

In [ ]:
with Raw(destination, mode='r') as raw:
    for name, value in raw.content.metadata.attrs.items():
        display(f"{name}: {value}")
    raw.content.metadata.tree(attrs=True)

## Check some random images

In [ ]:
with Raw(destination, mode='r') as raw:
    m = min(n_images, 5)
    rand_int = sample(range(n_images), m)
    rand_int.sort()
    phase_samples = raw.content.phase.images[rand_int].compute()
    amplitude_samples = raw.content.amplitude.images[rand_int].compute()
    hologram_samples = raw.content.hologram.images[rand_int].compute()
    
    fig0, axs0 = plt.subplots(1, m, figsize=(15, 3))
    for j, ax in enumerate(axs0):
        ax.imshow(hologram_samples[j])
        ax.set_title(f'Index: {rand_int[j]}')
    fig0.suptitle('Hologram', fontsize=16)

    fig1, axs1 = plt.subplots(1, m, figsize=(15, 3))
    for j, ax in enumerate(axs1):
        ax.imshow(phase_samples[j])
        ax.set_title(f'Index: {rand_int[j]}')
    fig1.suptitle('Phase', fontsize=16)

    fig2, axs2 = plt.subplots(1, m, figsize=(15, 3))
    for j, ax in enumerate(axs2):
        ax.imshow(amplitude_samples[j])
        ax.set_title(f'Index: {rand_int[j]}')
    fig2.suptitle('Amplitude', fontsize=16)

## Check backgorund estimation

In [ ]:
with Raw(destination, mode='r') as raw:
    phase_bg = raw.content.phase.statistics.images.background[...].compute()
    amplitude_bg = raw.content.amplitude.statistics.images.background[...].compute()
    hologram_bg = raw.content.hologram.statistics.images.background[...].compute()
    
    fig, axs = plt.subplots(1, 3, figsize=(15, 3))
    axs[0].imshow(phase_bg)
    axs[0].set_title('Phase')
    axs[1].imshow(amplitude_bg)
    axs[1].set_title('Amplitude')
    axs[2].imshow(hologram_bg)
    axs[2].set_title('Hologram')